In [ ]:
import torch

print("Pytorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU",torch.cuda.get_device_name(0))

device="cuda" if torch.cuda.is_available() else "cpu"

print("Using device:" , device)




In [ ]:
!nvidia-smi

In [ ]:
%pip install -q \
    transformers \
    datasets \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    huggingface_hub

In [ ]:
import transformers
import datasets
import accelerate
import evaluate
import jiwer
import librosa
import soundfile
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Evaluate:", evaluate.__version__)

print("Speech environment setup complete")

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicVoices",
    "telugu",
    split="valid",
    streaming=True
)

print(dataset)

In [ ]:
sample = next(iter(dataset))
print(sample)

In [ ]:
audio=sample["audio_filepath"]
audio_samples =  audio.get_all_samples()

waveform=audio_samples.data
sample_rate=audio_samples.sample_rate

print("Waveform shape:", waveform.shape)
print("Sample rate:", sample_rate)
print("Number of samples:", waveform.shape[-1])

In [ ]:
from IPython.display import Audio, display

display(
    Audio(
        waveform.squeeze().cpu().numpy(),
        rate=sample_rate
    )
)

In [ ]:
model_url = "https://indicwhisper.objectstore.e2enetworks.net/telugu_models.zip"
model_zip = "/content/telugu_models.zip"
model_dir = "/content/indicwhisper_telugu"

In [ ]:
!wget -q --show-progress "$model_url" -O "$model_zip"

In [ ]:
import os 
size_gb =os.path.getsize(model_zip) / (1024**3)

print(f"Downloaded archive size: {size_gb:.2f} GB")

In [ ]:
import zipfile

with zipfile.ZipFile(model_zip, 'r') as z:
    files = z.namelist()

print("Number of files:", len(files))

for file in files[:30]:
    print(file)

In [ ]:
import os 
import zipfile

with zipfile.ZipFile(model_zip, 'r') as z:
    z.extractall(model_dir)

checkpoint_dir= os.path.join(
    model_dir,
    "telugu_models",
    "whisper-medium-te_alldata_multigpu"
)

print("Checkpoint directory:",checkpoint_dir)
print("Exists:",os.path.exists(checkpoint_dir))


In [ ]:
important_files = [
    "config.json",
    "generation_config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "pytorch_model.bin",
]

for filename in important_files:
    path = os.path.join(checkpoint_dir, filename)
    print(filename, "->", os.path.exists(path))

In [ ]:
weights_path = os.path.join(
    checkpoint_dir,
    "pytorch_model.bin"
)

weights_size_gb = os.path.getsize(weights_path) / (1024 ** 3)

print(f"Model weights size: {weights_size_gb:.2f} GB")

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(checkpoint_dir)

print(processor)

In [ ]:
audio_array=waveform.squeeze().cpu().numpy()

inputs = processor(
    audio_array,
    sampling_rate = sample_rate,
    return_tensors="pt"


)
print("Input features shape:", inputs.input_features.shape)
print("Input features dtype:", inputs.input_features.dtype)

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    checkpoint_dir,
    dtype=torch.float16,
    low_cpu_mem_usage=True
)

model = model.to(device)
model.eval()

In [ ]:
num_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {num_params / 1e6:.1f} million")
print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

In [ ]:
gpu_memory_gb = torch.cuda.memory_allocated() / (1024 ** 3)

print(f"GPU memory currently allocated: {gpu_memory_gb:.2f} GB")

In [ ]:
forced_decoder_ids = processor.tokenizer.get_decoder_prompt_ids(
    language="te",
    task="transcribe"
)

print(forced_decoder_ids)

In [ ]:
input_features = inputs.input_features.to(
    device=device,
    dtype=torch.float16
)

print("Features device:", input_features.device)
print("Features dtype:", input_features.dtype)

In [ ]:
with torch.inference_mode():
    predicted_ids = model.generate(
        input_features,
        forced_decoder_ids=forced_decoder_ids
    )

In [ ]:
print("Predicted token IDs:")
print(predicted_ids)

In [ ]:
prediction = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0].strip()

reference = sample["normalized"]

print("Reference :", reference)
print("Prediction:", prediction)

In [ ]:
import os

print("dataset exists:", "dataset" in globals())
print("sample exists:", "sample" in globals())
print("processor exists:", "processor" in globals())
print("model exists:", "model" in globals())
print("predicted_ids exists:", "predicted_ids" in globals())

print(
    "checkpoint files exist:",
    os.path.exists(
        "/content/indicwhisper_telugu/telugu_models/whisper-medium-te_alldata_multigpu"
    )
)

In [ ]:
forced_decoder_ids = processor.tokenizer.get_decoder_prompt_ids(
    language="te",
    task="transcribe"
)

input_features = inputs.input_features.to(
    device=device,
    dtype=torch.float16
)

with torch.inference_mode():
    predicted_ids = model.generate(
        input_features,
        forced_decoder_ids=forced_decoder_ids
    )

prediction = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0].strip()

reference = sample["normalized"]

print("Reference :", reference)
print("Prediction:", prediction)

In [ ]:
from jiwer import process_words

def normalize_for_wer(text):
    return " ".join(text.strip().split())

reference_clean = normalize_for_wer(reference)
prediction_clean = normalize_for_wer(prediction)

result = process_words(
    reference_clean,
    prediction_clean
)

print("WER:", result.wer)
print("WER %:", result.wer * 100)
print("Substitutions:", result.substitutions)
print("Deletions:", result.deletions)
print("Insertions:", result.insertions)
print("Correct words:", result.hits)

In [ ]:
def transcribe_sample(sample):
    audio = sample["audio_filepath"].get_all_samples()

    waveform = audio.data.squeeze().cpu().numpy()
    sample_rate = audio.sample_rate

    inputs = processor(
        waveform,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device=device,
        dtype=torch.float16
    )

    with torch.inference_mode():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids
        )

    prediction = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0].strip()

    return prediction

In [ ]:
from itertools import islice

test_samples = list(islice(dataset, 20))

print("Samples collected:", len(test_samples))

In [ ]:
results = []

for i, sample_item in enumerate(test_samples):
    reference = normalize_for_wer(sample_item["normalized"])

    prediction = transcribe_sample(sample_item)
    prediction = normalize_for_wer(prediction)

    score = process_words(
        reference,
        prediction
    )

    results.append({
        "index": i,
        "speaker_id": sample_item["speaker_id"],
        "duration": sample_item["duration"],
        "reference": reference,
        "prediction": prediction,
        "wer": score.wer,
        "substitutions": score.substitutions,
        "deletions": score.deletions,
        "insertions": score.insertions
    })

    print(f"{i + 1}/20 complete")

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df.head()

In [ ]:
for i in range(5):
    print("REFERENCE :", results_df.iloc[i]["reference"])
    print("PREDICTION:", results_df.iloc[i]["prediction"])
    print("WER       :", results_df.iloc[i]["wer"])
    print()

In [ ]:
references = results_df["reference"].tolist()
predictions = results_df["prediction"].tolist()

corpus_result = process_words(
    references,
    predictions
)

reference_words = (
    corpus_result.hits
    + corpus_result.substitutions
    + corpus_result.deletions
)

print("Reference words:", reference_words)
print("Correct words:", corpus_result.hits)
print("Substitutions:", corpus_result.substitutions)
print("Deletions:", corpus_result.deletions)
print("Insertions:", corpus_result.insertions)

print("Corpus WER:", corpus_result.wer)
print("Corpus WER %:", corpus_result.wer * 100)

In [ ]:
print(dataset)
print(dataset.info)

In [ ]:
print(dataset.info.splits)

In [ ]:
num_samples = dataset.info.splits["valid"].num_examples

print("IndicVoices Telugu validation samples:", num_samples)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

results_dir = Path(
    "/content/drive/MyDrive/DhwaniLab/results/indicwhisper"
)

results_dir.mkdir(
    parents=True,
    exist_ok=True
)

checkpoint_file = results_dir / "indicvoices_telugu_valid.csv"

print("Results directory:", results_dir)
print("Checkpoint file:", checkpoint_file)

In [ ]:
results_df.to_csv(
    checkpoint_file,
    index=False
)

print("Saved rows:", len(results_df))
print("Saved to:", checkpoint_file)

In [ ]:
saved_df = pd.read_csv(checkpoint_file)

print("Rows recovered:", len(saved_df))
saved_df.head()

In [ ]:
saved_df = pd.read_csv(checkpoint_file)

completed_indices = set(
    saved_df["index"].astype(int).tolist()
)

all_results = saved_df.to_dict("records")

print("Already completed:", len(completed_indices))
print("Remaining:", num_samples - len(completed_indices))

In [ ]:
save_every = 25
new_since_save = 0

for index, sample_item in enumerate(dataset):

    if index in completed_indices:
        continue

    reference = normalize_for_wer(
        sample_item["normalized"]
    )

    prediction = transcribe_sample(
        sample_item
    )

    prediction = normalize_for_wer(
        prediction
    )

    score = process_words(
        reference,
        prediction
    )

    row = {
        "index": index,
        "speaker_id": sample_item["speaker_id"],
        "duration": sample_item["duration"],
        "reference": reference,
        "prediction": prediction,
        "wer": score.wer,
        "substitutions": score.substitutions,
        "deletions": score.deletions,
        "insertions": score.insertions,
    }

    all_results.append(row)
    completed_indices.add(index)

    new_since_save += 1

    print(
        f"{index + 1}/{num_samples} | "
        f"WER: {score.wer:.3f}"
    )

    if new_since_save >= save_every:

        checkpoint_df = pd.DataFrame(
            all_results
        ).sort_values("index")

        checkpoint_df.to_csv(
            checkpoint_file,
            index=False
        )

        print(
            f"Checkpoint saved: "
            f"{len(checkpoint_df)} samples"
        )

        new_since_save = 0

In [ ]:
import os
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint_dir = (
    "/content/indicwhisper_telugu/"
    "telugu_models/"
    "whisper-medium-te_alldata_multigpu"
)

print("IndicWhisper files still exist:", os.path.exists(checkpoint_dir))

In [ ]:
num_samples = dataset.info.splits["valid"].num_examples

print("Total samples:", num_samples)

In [ ]:
names = [
    "dataset",
    "processor",
    "model",
    "forced_decoder_ids",
    "normalize_for_wer",
    "transcribe_sample",
    "num_samples",
    "checkpoint_file",
    "completed_indices",
    "all_results",
]

for name in names:
    print(name, ":", name in globals())

In [ ]:
saved_df = pd.read_csv(checkpoint_file)

completed_indices = set(
    saved_df["index"].astype(int).tolist()
)

all_results = saved_df.to_dict("records")

print("Saved rows:", len(saved_df))
print("Last saved index:", saved_df["index"].max())
print("Remaining:", num_samples - len(completed_indices))

In [ ]:
final_df = pd.read_csv(checkpoint_file)

references = final_df["reference"].tolist()
predictions = final_df["prediction"].tolist()

final_result = process_words(
    references,
    predictions
)

reference_words = (
    final_result.hits
    + final_result.substitutions
    + final_result.deletions
)

print("Samples:", len(final_df))
print("Reference words:", reference_words)
print("Correct words:", final_result.hits)
print("Substitutions:", final_result.substitutions)
print("Deletions:", final_result.deletions)
print("Insertions:", final_result.insertions)
print("Corpus WER:", final_result.wer)
print("Corpus WER %:", final_result.wer * 100)